In [1]:
from pypdf import PdfReader



def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""

    for page in reader.pages:
        content = page.extract_text()
        if content:
            text += content
    
    return text

def chunk_text(text,chunk_size,overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start : end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks
pdf_text = load_pdf("Some_Useful_things_about_ML.pdf")

chunks = chunk_text(pdf_text,overlap = 100,chunk_size = 400)



In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14334.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:

chunk_embeddings = model.encode(chunks)

print(type(chunk_embeddings))
print(len(chunk_embeddings))        
print(len(chunk_embeddings[0]))   




<class 'numpy.ndarray'>
172
384


In [4]:
import faiss
import numpy as np

In [5]:
embeddings = np.array(chunk_embeddings).astype("float32")

In [6]:
dimension = embeddings.shape[1]  # should be 384

index = faiss.IndexFlatL2(dimension)

In [7]:
index.add(embeddings)

In [25]:
query = "What does the article say about “folk knowledge” in machine learning?"
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")
k = 3  # top results

distances, indices = index.search(query_embedding, k)
retrieved_chunks = [chunks[i] for i in indices[0]]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Result {i+1} ---\n")
    print(chunk)

import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)   # remove weird spacing
    text = re.sub(r'\n+', ' ', text)   # remove line breaks
    return text

pdf_text = clean_text(pdf_text)


--- Result 1 ---

eded to successfully develop 
machine learning applications is not 
readily available in them. As a result, 
many machine learning projects take 
much longer than necessary or wind 
up producing less-than-ideal results. 
Yet much of this folk knowledge is 
fairly easy to communicate. This is 
the purpose of this article.
doi:10.1145/2347736.2347755
Tapping into the “folk knowledge” needed to 
adva

--- Result 2 ---

rpose of this article.
doi:10.1145/2347736.2347755
Tapping into the “folk knowledge” needed to 
advance machine learning applications.
by Pedro domingos
a f ew u seful 
t
hings to 
Know 
a
bout 
m
achine 
Learning
 key insights
    machine learning algorithms can figure 
out how to perform impor
tant tasks 
by generalizing from examples. 
t
his is 
often feasible and cost-effective where 
manual p

--- Result 3 ---

y 
the philosopher David Hume over 200 
years ago, but even today many mis-
takes in machine learning stem from 
failing to appreciate it. Ev

In [ ]:
import google.generativeai as genai

genai.configure(api_key="API_KEY")

model_llm = genai.GenerativeModel("gemini-flash-lite-latest")




In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
You are an AI assistant.

Use ONLY the context below to answer the question.
If the answer is not present, say "I don't know".
Answer in clear bullet points.
Context:
{context}

Question:
{query}

Answer:
"""
    return prompt

In [28]:
prompt = build_prompt(query, retrieved_chunks)

response = model_llm.generate_content(prompt)

print(response.text)

The purpose of the article is to communicate the "folk knowledge" needed to advance machine learning applications, because the knowledge needed to successfully develop machine learning applications is not readily available, causing many projects to take too long or produce less-than-ideal results, even though much of this folk knowledge is fairly easy to communicate.


In [29]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

s1 = "Machine learning is used in healthcare"
s2 = "AI is applied in medicine"
s3 = "The sky is blue"

embeddings = model.encode([s1, s2, s3])

sim_1_2 = cosine_similarity([embeddings[0]], [embeddings[1]])
sim_1_3 = cosine_similarity([embeddings[0]], [embeddings[2]])

print("Similar meaning:", sim_1_2)
print("Different meaning:", sim_1_3)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3209.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similar meaning: [[0.5808526]]
Different meaning: [[0.04598131]]


In [30]:
s1 = "ML is used for fraud detection"
s2 = "Machine learning helps detect fraud"

embeddings = model.encode([s1, s2])

print(cosine_similarity([embeddings[0]], [embeddings[1]]))

[[0.6716368]]


In [37]:
s1 = "Machine learning is used in healthcare"
s2 = "Machine learning is used in healthcare and medicine and surgery"

embeddings = model.encode([s1, s2])

print(cosine_similarity([embeddings[0]], [embeddings[1]]))

[[0.91780686]]


In [38]:
chunk = "Machine learning is used in healthcare. The sky is blue."

query = "applications of machine learning"

embeddings = model.encode([chunk, query])

print(cosine_similarity([embeddings[0]], [embeddings[1]]))

[[0.54560375]]


In [43]:
small = "Machine Learning is used in healthcare"
big = "Machine learning is used in healthcare, finance, marketing, sports, gaming, weather prediction, and many unrelated fields"

query = "machine learning healthcare"

embeddings = model.encode([small, big, query])

print("Small chunk:", cosine_similarity([embeddings[0]], [embeddings[2]]))
print("Big chunk:", cosine_similarity([embeddings[1]], [embeddings[2]]))

Small chunk: [[0.8595584]]
Big chunk: [[0.6181892]]


In [44]:
s1 = "bank of a river"
s2 = "financial bank account"

embeddings = model.encode([s1, s2])

print(cosine_similarity([embeddings[0]], [embeddings[1]]))

[[0.46749702]]
